In [9]:
# Install nba_api
!pip install nba_api

from nba_api.stats.endpoints import scoreboardv2, boxscoretraditionalv2
from datetime import datetime, timedelta
import pandas as pd
import io

# Function to get yesterday's date
def get_yesterday():
    return (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')

# Function to fetch yesterday's game IDs
def get_yesterdays_game_ids():
    yesterday = get_yesterday()
    scoreboard = scoreboardv2.ScoreboardV2(game_date=yesterday)
    games = scoreboard.get_normalized_dict()['GameHeader']
    return [game['GAME_ID'] for game in games]

# Function to fetch box scores for a game
def fetch_box_scores(game_id):
    boxscore = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id)
    boxscore_data = boxscore.get_normalized_dict()['PlayerStats']
    return pd.DataFrame(boxscore_data)

# Fetch all box scores for yesterday's games
def get_yesterdays_box_scores():
    game_ids = get_yesterdays_game_ids()
    all_box_scores = []
    for game_id in game_ids:
        box_scores = fetch_box_scores(game_id)
        all_box_scores.append(box_scores)
    return pd.concat(all_box_scores, ignore_index=True)

# Simulate CSV Paste for Predicted Stats
print("Paste your CSV data below (then press Enter twice):")
csv_input = """"""  # Paste CSV data here
while True:
    line = input()
    if line.strip() == "":
        break
    csv_input += line + "\n"

# Convert CSV input to a DataFrame
try:
    print("\nRaw CSV Input:")
    print(csv_input)  # Debugging step

    # Define headers if necessary
    headers = [
        'index', 'Player', 'Team', 'PPG', 'PPG SD', 'PPG CV',
        'RPG', 'RPG SD', 'RPG CV', 'APG', 'APG SD', 'APG CV',
        'Adjusted PPG', 'Adjusted RPG', 'Adjusted APG'
    ]

    predicted_data = pd.read_csv(io.StringIO(csv_input), names=headers, skiprows=1)
    print("\nParsed Predicted DataFrame:")
    print(predicted_data.head())  # Debugging step

except Exception as e:
    print("Error processing CSV data. Ensure the format is correct.")
    raise e

# Check for required columns
required_columns = ['Player', 'Adjusted PPG', 'Adjusted RPG', 'Adjusted APG']
missing_columns = [col for col in required_columns if col not in predicted_data.columns]
if missing_columns:
    print(f"\nColumns in DataFrame: {list(predicted_data.columns)}")
    print(f"Required Columns: {required_columns}")
    raise ValueError(f"Missing columns in predicted data: {', '.join(missing_columns)}")

# Fetch actual box scores
actual_box_scores = get_yesterdays_box_scores()

# Filter relevant columns from actual data
actual_data = actual_box_scores[['PLAYER_NAME', 'PTS', 'REB', 'AST']].rename(
    columns={"PLAYER_NAME": "Player", "PTS": "Actual_PPG", "REB": "Actual_RPG", "AST": "Actual_APG"}
)

# Merge predicted and actual data
comparison = pd.merge(predicted_data, actual_data, on="Player", how="inner")

# Calculate errors
comparison['Error_PPG'] = abs(comparison['Adjusted PPG'] - comparison['Actual_PPG'])
comparison['Error_RPG'] = abs(comparison['Adjusted RPG'] - comparison['Actual_RPG'])
comparison['Error_APG'] = abs(comparison['Adjusted APG'] - comparison['Actual_APG'])

# Calculate MAE and RMSE
mae = comparison[['Error_PPG', 'Error_RPG', 'Error_APG']].mean().mean()
rmse = ((comparison[['Error_PPG', 'Error_RPG', 'Error_APG']]**2).mean().mean())**0.5

# Define thresholds for assessment
MAE_THRESHOLD = 5.0  # Adjust these based on your use case
RMSE_THRESHOLD = 6.0

# Assess performance
mae_quality = "Good" if mae <= MAE_THRESHOLD else "Bad"
rmse_quality = "Good" if rmse <= RMSE_THRESHOLD else "Bad"

print("\nComparison Data:")
print(comparison)
print(f"\nMean Absolute Error (MAE): {mae} ({mae_quality})")
print(f"Root Mean Squared Error (RMSE): {rmse} ({rmse_quality})")

# Performance message
if mae_quality == "Good" and rmse_quality == "Good":
    print("\nPerformance: The predictions are accurate.")
elif mae_quality == "Bad" and rmse_quality == "Bad":
    print("\nPerformance: The predictions are not very accurate. Consider refining your model.")
else:
    print("\nPerformance: Mixed results. One metric is accurate, while the other needs improvement.")


Paste your CSV data below (then press Enter twice):
index,Player,Team,PPG,PPG SD,PPG CV,RPG,RPG SD,RPG CV,APG,APG SD,APG CV,Adjusted PPG,Adjusted RPG,Adjusted APG 0,Ty Jerome,CLE,11.76,7.48,0.64,1.95,1.6,0.82,3.62,2.18,0.6,12.73,2.11,3.92 1,Caris LeVert,CLE,11.5,4.94,0.43,2.94,2.08,0.71,4.0,2.03,0.51,12.45,3.18,4.33 2,Evan Mobley,CLE,17.76,5.08,0.29,9.33,3.06,0.33,2.76,1.67,0.6,19.23,10.11,2.99 3,Sam Merrill,CLE,6.94,4.58,0.66,1.72,1.32,0.77,1.78,1.4,0.79,7.52,1.86,1.92 4,Craig Porter Jr.,CLE,2.75,3.92,1.43,0.88,1.2,1.38,1.38,1.78,1.3,2.98,0.95,1.49 5,Darius Garland,CLE,20.33,8.62,0.42,2.43,1.43,0.59,6.71,2.35,0.35,22.01,2.63,7.27 6,Tristan Thompson,CLE,3.25,2.82,0.87,2.12,3.04,1.43,0.5,0.76,1.51,3.52,2.3,0.54 7,JT Thor,CLE,7.0,7.07,1.01,1.0,1.41,1.41,0.0,0.0,0.0,7.58,1.08,0.0 8,Georges Niang,CLE,8.55,4.64,0.54,3.64,2.08,0.57,1.18,0.85,0.72,9.25,3.94,1.28 9,Jaylon Tyson,CLE,2.5,4.5,1.8,2.0,2.98,1.49,0.92,1.98,2.15,2.71,2.17,0.99 10,Jarrett Allen,CLE,14.05,6.09,0.43,10.32,4.05,0.39,1.59

In [ ]:
# Install nba_api
!pip install nba_api

import pandas as pd
import matplotlib.pyplot as plt
import io
from nba_api.stats.endpoints import scoreboardv2, boxscoretraditionalv2
from datetime import datetime, timedelta

# Function to get yesterday's date
def get_yesterday():
    return (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')

# Function to fetch yesterday's game IDs
def get_yesterdays_game_ids():
    yesterday = get_yesterday()
    scoreboard = scoreboardv2.ScoreboardV2(game_date=yesterday)
    games = scoreboard.get_normalized_dict()['GameHeader']
    return [game['GAME_ID'] for game in games]

# Function to fetch box scores for a game
def fetch_box_scores(game_id):
    boxscore = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id)
    boxscore_data = boxscore.get_normalized_dict()['PlayerStats']
    return pd.DataFrame(boxscore_data)

# Fetch all box scores for yesterday's games
def get_yesterdays_box_scores():
    game_ids = get_yesterdays_game_ids()
    all_box_scores = []
    for game_id in game_ids:
        box_scores = fetch_box_scores(game_id)
        all_box_scores.append(box_scores)
    return pd.concat(all_box_scores, ignore_index=True)

# Simulate CSV Paste for Predicted Stats
print("Paste your CSV data below (then press Enter twice):")
csv_input = """"""  # Paste CSV data here
while True:
    line = input()
    if line.strip() == "":
        break
    csv_input += line + "\n"

# Convert CSV input to a DataFrame
try:
    print("\nRaw CSV Input:")
    print(csv_input)  # Debugging step

    # Define headers if necessary
    headers = [
        'index', 'Player', 'Team', 'PPG', 'PPG SD', 'PPG CV', 'PPG Diff',
        'RPG', 'RPG SD', 'RPG CV', 'RPG Diff',
        'APG', 'APG SD', 'APG CV', 'APG Diff',
        'Adjusted PPG', 'Adjusted RPG', 'Adjusted APG'
    ]

    predicted_data = pd.read_csv(io.StringIO(csv_input), names=headers, skiprows=1)
    print("\nParsed Predicted DataFrame:")
    print(predicted_data.head())  # Debugging step

except Exception as e:
    print("Error processing CSV data. Ensure the format is correct.")
    raise e

# Check for required columns
required_columns = [
    'Player', 'PPG Diff', 'RPG Diff', 'APG Diff',
    'Adjusted PPG', 'Adjusted RPG', 'Adjusted APG'
]
missing_columns = [col for col in required_columns if col not in predicted_data.columns]
if missing_columns:
    print(f"\nColumns in DataFrame: {list(predicted_data.columns)}")
    print(f"Required Columns: {required_columns}")
    raise ValueError(f"Missing columns in predicted data: {', '.join(missing_columns)}")

# Fetch actual box scores
actual_box_scores = get_yesterdays_box_scores()

# Filter relevant columns from actual data
actual_data = actual_box_scores[['PLAYER_NAME', 'PTS', 'REB', 'AST']].rename(
    columns={"PLAYER_NAME": "Player", "PTS": "Actual_PPG", "REB": "Actual_RPG", "AST": "Actual_APG"}
)

# Merge predicted and actual data
comparison = pd.merge(predicted_data, actual_data, on="Player", how="inner")

# Calculate errors for Adjusted stats
comparison['Error_Adjusted_PPG'] = abs(comparison['Adjusted PPG'] - comparison['Actual_PPG'])
comparison['Error_Adjusted_RPG'] = abs(comparison['Adjusted RPG'] - comparison['Actual_RPG'])
comparison['Error_Adjusted_APG'] = abs(comparison['Adjusted APG'] - comparison['Actual_APG'])

# Calculate errors for Diff stats
comparison['Error_PPG_Diff'] = abs(comparison['PPG Diff'] - comparison['Actual_PPG'])
comparison['Error_RPG_Diff'] = abs(comparison['RPG Diff'] - comparison['Actual_RPG'])
comparison['Error_APG_Diff'] = abs(comparison['APG Diff'] - comparison['Actual_APG'])

# Calculate MAE and RMSE for Adjusted stats
mae_adjusted = comparison[['Error_Adjusted_PPG', 'Error_Adjusted_RPG', 'Error_Adjusted_APG']].mean()
rmse_adjusted = (comparison[['Error_Adjusted_PPG', 'Error_Adjusted_RPG', 'Error_Adjusted_APG']]**2).mean()**0.5

# Calculate MAE and RMSE for Diff stats
mae_diff = comparison[['Error_PPG_Diff', 'Error_RPG_Diff', 'Error_APG_Diff']].mean()
rmse_diff = (comparison[['Error_PPG_Diff', 'Error_RPG_Diff', 'Error_APG_Diff']]**2).mean()**0.5

# Compare Adjusted vs Diff
better_mae = "Adjusted" if mae_adjusted.mean() < mae_diff.mean() else "Diff"
better_rmse = "Adjusted" if rmse_adjusted.mean() < rmse_diff.mean() else "Diff"

print("\nComparison Data:")
print(comparison)

print("\n" + "="*50)
print("**Performance Metrics for Adjusted Stats**")
print("="*50)
print(f"PPG:  MAE = {mae_adjusted['Error_Adjusted_PPG']:.2f}, RMSE = {rmse_adjusted['Error_Adjusted_PPG']:.2f}")
print(f"RPG:  MAE = {mae_adjusted['Error_Adjusted_RPG']:.2f}, RMSE = {rmse_adjusted['Error_Adjusted_RPG']:.2f}")
print(f"APG:  MAE = {mae_adjusted['Error_Adjusted_APG']:.2f}, RMSE = {rmse_adjusted['Error_Adjusted_APG']:.2f}")

print("\n" + "="*50)
print("**Performance Metrics for Diff Stats**")
print("="*50)
print(f"PPG:  MAE = {mae_diff['Error_PPG_Diff']:.2f}, RMSE = {rmse_diff['Error_PPG_Diff']:.2f}")
print(f"RPG:  MAE = {mae_diff['Error_RPG_Diff']:.2f}, RMSE = {rmse_diff['Error_RPG_Diff']:.2f}")
print(f"APG:  MAE = {mae_diff['Error_APG_Diff']:.2f}, RMSE = {rmse_diff['Error_APG_Diff']:.2f}")

print("\n" + "="*50)
print("**Overall Metrics Comparison**")
print("="*50)
print(f"Overall MAE - Adjusted: {mae_adjusted.mean():.2f}, Diff: {mae_diff.mean():.2f} -> Better: {better_mae}")
print(f"Overall RMSE - Adjusted: {rmse_adjusted.mean():.2f}, Diff: {rmse_diff.mean():.2f} -> Better: {better_rmse}")

# Visualization of Errors
plt.figure(figsize=(15, 10))

# PPG Errors
plt.subplot(3, 1, 1)
plt.bar(comparison['Player'], comparison['Error_Adjusted_PPG'], color='blue', alpha=0.6, label='Adjusted PPG Error')
plt.bar(comparison['Player'], comparison['Error_PPG_Diff'], color='orange', alpha=0.6, label='PPG Diff Error')
plt.title('PPG Errors')
plt.ylabel('Error')
plt.legend()
plt.xticks(rotation=90)

# RPG Errors
plt.subplot(3, 1, 2)
plt.bar(comparison['Player'], comparison['Error_Adjusted_RPG'], color='blue', alpha=0.6, label='Adjusted RPG Error')
plt.bar(comparison['Player'], comparison['Error_RPG_Diff'], color='orange', alpha=0.6, label='RPG Diff Error')
plt.title('RPG Errors')
plt.ylabel('Error')
plt.legend()
plt.xticks(rotation=90)

# APG Errors
plt.subplot(3, 1, 3)
plt.bar(comparison['Player'], comparison['Error_Adjusted_APG'], color='blue', alpha=0.6, label='Adjusted APG Error')
plt.bar(comparison['Player'], comparison['Error_APG_Diff'], color='orange', alpha=0.6, label='APG Diff Error')
plt.title('APG Errors')
plt.ylabel('Error')
plt.legend()
plt.xticks(rotation=90)

plt.tight_layout()
plt.show()
